# Early Behavioral Dropout Risk Forecasting in Online Learning Environments
## Notebook 01: Data Understanding & Exploratory Data Analysis (EDA)
### 🎯 Project Objective
The objective of this project is to identify students who are at risk of dropping out from an online course at an early stage using demographic and behavioral learning data from the Open University Learning Analytics Dataset (OULAD).
This notebook focuses on understanding the dataset, exploring its structure, identifying data quality issues, and performing exploratory data analysis before feature engineering and model development.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [6]:
import os
os.getcwd()

'C:\\Users\\user'

In [7]:
import os
os.chdir(r"C:\Users\user\OneDrive\Desktop\Ghost-Student-Dropout-Prediction")
print(os.getcwd())

C:\Users\user\OneDrive\Desktop\Ghost-Student-Dropout-Prediction


In [8]:
courses = pd.read_csv("data/courses.csv")
studentInfo = pd.read_csv("data/studentInfo.csv")
studentAssessment = pd.read_csv("data/studentAssessment.csv")
studentRegistration = pd.read_csv("data/studentRegistration.csv")
studentVle = pd.read_csv("data/studentVle.csv")
vle = pd.read_csv("data/vle.csv")
assessments = pd.read_csv("data/assessments.csv")

In [9]:
print("All datasets loaded successfully.")

All datasets loaded successfully.


In [10]:
datasets = {
    "courses": courses,
    "studentInfo": studentInfo,
    "studentAssessment": studentAssessment,
    "studentRegistration": studentRegistration,
    "studentVle": studentVle,
    "vle": vle,
    "assessments": assessments
}
summary = pd.DataFrame({
    "Dataset": datasets.keys(),
    "Rows": [data.shape[0] for data in datasets.values()],
    "Columns": [data.shape[1] for data in datasets.values()]
})
summary

,Dataset,Rows,Columns
0,courses,22,3
1,studentInfo,32593,12
2,studentAssessment,173912,5
3,studentRegistration,32593,5
4,studentVle,10655280,6
5,vle,6364,6
6,assessments,206,6


## Understanding the Student Information Dataset

In [12]:
studentInfo.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


In [13]:
studentInfo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   code_module           32593 non-null  object
 1   code_presentation     32593 non-null  object
 2   id_student            32593 non-null  int64 
 3   gender                32593 non-null  object
 4   region                32593 non-null  object
 5   highest_education     32593 non-null  object
 6   imd_band              32593 non-null  object
 7   age_band              32593 non-null  object
 8   num_of_prev_attempts  32593 non-null  int64 
 9   studied_credits       32593 non-null  int64 
 10  disability            32593 non-null  object
 11  final_result          32593 non-null  object
dtypes: int64(3), object(9)
memory usage: 3.0+ MB


In [14]:
studentInfo["id_student"].nunique()

28785

In [15]:
studentInfo["id_student"].duplicated().sum()

np.int64(3808)

In [16]:
studentInfo.describe(include="all")

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
count,32593,32593,3.259300e+04,32593,32593,32593,32593,32593,32593.000000,32593.000000,32593,32593
unique,7,4,NaN,2,13,5,11,3,NaN,NaN,2,4
top,BBB,2014J,NaN,M,Scotland,A Level or Equivalent,20-30%,0-35,NaN,NaN,N,Pass
freq,7909,11260,NaN,17875,3446,14045,3654,22944,NaN,NaN,29429,12361
mean,NaN,NaN,7.066877e+05,NaN,NaN,NaN,NaN,NaN,0.163225,79.758691,NaN,NaN
std,NaN,NaN,5.491673e+05,NaN,NaN,NaN,NaN,NaN,0.479758,41.071900,NaN,NaN
min,NaN,NaN,3.733000e+03,NaN,NaN,NaN,NaN,NaN,0.000000,30.000000,NaN,NaN
25%,NaN,NaN,5.085730e+05,NaN,NaN,NaN,NaN,NaN,0.000000,60.000000,NaN,NaN
50%,NaN,NaN,5.903100e+05,NaN,NaN,NaN,NaN,NaN,0.000000,60.000000,NaN,NaN
75%,NaN,NaN,6.444530e+05,NaN,NaN,NaN,NaN,NaN,0.000000,120.000000,NaN,NaN


### Exploring Numerical Features

In [17]:
studentInfo["studied_credits"].value_counts().sort_index()

studied_credits
30     3749
40       22
45       36
50        7
55        3
       ... 
480       1
540       1
585       1
630       1
655       1
Name: count, Length: 61, dtype: int64

In [18]:
studentInfo["studied_credits"].value_counts()

studied_credits
60     16751
120     6328
30      3749
90      3144
180      830
       ...  
480        1
655        1
390        1
235        1
430        1
Name: count, Length: 61, dtype: int64

The `studied_credits` feature contains several rare values, including 655 credits. Although these values appear infrequently, they should not be treated as invalid without domain verification. Statistical outlier detection will be performed later, but business context will also be considered before removing any observations.

### Key Findings from studentInfo
1->The studentInfo dataset contains 32,593 enrollment records and 12 features, providing demographic and academic information about students.
2->There are 28,785 unique students, indicating that some students enrolled in multiple course presentations. Therefore, each row represents an enrollment record rather than a unique student.
3->No missing values were found in any column, so this dataset does not require missing value treatment.
4->The dataset contains 9 categorical and 3 numerical features. The categorical features will require encoding before model training.
5->The final_result column contains four categories (Pass, Fail, Distinction, Withdrawn), making it a suitable candidate for defining the target variable in the dropout prediction problem.
6->The studied_credits feature contains several rare values (such as 655 credits). These values should be verified using both statistical techniques and business understanding before deciding whether they are outliers.
7->Features such as gender, region, highest_education, age_band, and num_of_prev_attempts appear to be potentially useful predictors, but their actual impact will be verified during exploratory data analysis and feature engineering.

## Understanding the Student Registration Dataset

In [19]:
studentRegistration.head()

,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159,?
1,AAA,2013J,28400,-53,?
2,AAA,2013J,30268,-92,12
3,AAA,2013J,31604,-52,?
4,AAA,2013J,32885,-176,?


In [20]:
studentRegistration.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   code_module          32593 non-null  object
 1   code_presentation    32593 non-null  object
 2   id_student           32593 non-null  int64 
 3   date_registration    32593 non-null  object
 4   date_unregistration  32593 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.2+ MB


In [21]:
studentRegistration.describe(include="all")

,code_module,code_presentation,id_student,date_registration,date_unregistration
count,32593,32593,3.259300e+04,32593,32593
unique,7,4,NaN,333,417
top,BBB,2014J,NaN,-22,?
freq,7909,11260,NaN,1034,22521
mean,NaN,NaN,7.066877e+05,NaN,NaN
std,NaN,NaN,5.491673e+05,NaN,NaN
min,NaN,NaN,3.733000e+03,NaN,NaN
25%,NaN,NaN,5.085730e+05,NaN,NaN
50%,NaN,NaN,5.903100e+05,NaN,NaN
75%,NaN,NaN,6.444530e+05,NaN,NaN


In [22]:
studentRegistration["date_unregistration"].value_counts().head(10)

date_unregistration
?     22521
12      788
0       419
27      170
-1      147
-7       99
-2       99
10       91
-5       90
-8       87
Name: count, dtype: int64

In [23]:
studentRegistration["date_registration"].value_counts().sort_index().head(10)

date_registration
-1       58
-10     208
-100    208
-101    164
-102    245
-103    117
-104     22
-105     26
-106    185
-107    180
Name: count, dtype: int64

### Key Findings from studentRegistration
- The dataset contains one registration record for each enrollment.
- The `date_registration` and `date_unregistration` columns are currently stored as object data types because `date_unregistration` contains the symbol `?`.
- Registration dates are represented using relative values rather than calendar dates. Negative values likely indicate registration before the official course start.
- Most students (`22,521` records) have `?` in the `date_unregistration` column, suggesting that many students did not unregister during the course.
- The `date_unregistration` feature may introduce data leakage because it directly indicates when a student left the course. This feature will be evaluated carefully before model training.

## understanding studentAssessment

In [24]:
studentAssessment.head()

,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78
1,1752,28400,22,0,70
2,1752,31604,17,0,72
3,1752,32885,26,0,69
4,1752,38053,19,0,79


In [25]:
studentAssessment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173912 entries, 0 to 173911
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   id_assessment   173912 non-null  int64 
 1   id_student      173912 non-null  int64 
 2   date_submitted  173912 non-null  int64 
 3   is_banked       173912 non-null  int64 
 4   score           173912 non-null  object
dtypes: int64(4), object(1)
memory usage: 6.6+ MB


In [26]:
studentAssessment.describe(include="all")

,id_assessment,id_student,date_submitted,is_banked,score
count,173912.000000,1.739120e+05,173912.000000,173912.000000,173912
unique,NaN,NaN,NaN,NaN,102
top,NaN,NaN,NaN,NaN,100
freq,NaN,NaN,NaN,NaN,18813
mean,26553.803556,7.051507e+05,116.032942,0.010977,NaN
std,8829.784254,5.523952e+05,71.484148,0.104194,NaN
min,1752.000000,6.516000e+03,-11.000000,0.000000,NaN
25%,15022.000000,5.044290e+05,51.000000,0.000000,NaN
50%,25359.000000,5.852080e+05,116.000000,0.000000,NaN
75%,34883.000000,6.344980e+05,173.000000,0.000000,NaN


In [27]:
studentAssessment["score"].unique()[:20]

array(['78', '70', '72', '69', '79', '71', '68', '73', '67', '83', '66',
       '59', '82', '60', '75', '74', '62', '63', '84', '80'], dtype=object)

In [28]:
studentAssessment["score"].unique()

array(['78', '70', '72', '69', '79', '71', '68', '73', '67', '83', '66',
       '59', '82', '60', '75', '74', '62', '63', '84', '80', '76', '85',
       '57', '81', '87', '77', '45', '65', '61', '52', '54', '51', '88',
       '58', '64', '55', '38', '91', '47', '89', '36', '86', '49', '53',
       '39', '?', '90', '50', '56', '30', '11', '40', '94', '48', '46',
       '25', '34', '42', '18', '37', '28', '33', '95', '35', '44', '41',
       '15', '0', '43', '93', '32', '92', '98', '24', '19', '27', '29',
       '20', '97', '23', '99', '100', '10', '5', '13', '26', '22', '8',
       '12', '16', '9', '96', '14', '21', '17', '31', '6', '1', '7', '4',
       '2', '3'], dtype=object)

### Key Findings from `studentAssessment`

- The dataset contains **173,912 assessment submission records**, indicating that each student can have multiple assessment attempts.
- No missing values are present in any column.
- The `score` column is stored as an object data type and requires further investigation before preprocessing.
- The `date_submitted` feature contains both negative and positive values, suggesting that dates are recorded relative to the course timeline rather than as calendar dates.
- The `is_banked` feature appears to be a binary variable and its business meaning will be investigated before feature engineering.

## understanding Assessment.csv

In [29]:
assessments.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19,10.0
1,AAA,2013J,1753,TMA,54,20.0
2,AAA,2013J,1754,TMA,117,20.0
3,AAA,2013J,1755,TMA,166,20.0
4,AAA,2013J,1756,TMA,215,30.0


`studentAssessment` and `assessments` are connected through `id_assessment`.  
The `studentAssessment` table stores student-level submission and score information, while the `assessments` table stores assessment-level metadata such as module, presentation, type, date, and weight.

## understanding studentvle.csv


In [32]:
studentVle.head()

,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546614,-10,11
4,AAA,2013J,28400,546714,-10,1


### Key Findings from `studentVle`

- `studentVle` is the largest dataset in the project and contains detailed student interaction logs.
- Each row represents a student's interaction with a specific learning resource on a particular day.
- The `sum_click` feature measures the level of interaction with learning materials and is expected to be an important behavioral feature for dropout prediction.
- The dataset contains multiple records for the same student, indicating that behavioral features will need to be aggregated before model training.
- This dataset is expected to contribute the most important behavioral information for early dropout prediction.

## understanding vle.csv

In [33]:
vle.head()

,id_site,code_module,code_presentation,activity_type,week_from,week_to
0,546943,AAA,2013J,resource,?,?
1,546712,AAA,2013J,oucontent,?,?
2,546998,AAA,2013J,resource,?,?
3,546888,AAA,2013J,url,?,?
4,547035,AAA,2013J,resource,?,?


### Key Findings from `vle`

- The `vle` dataset contains metadata about learning resources available in the virtual learning environment.
- Each learning resource is uniquely identified by `id_site`.
- The `activity_type` column describes the type of learning resource, such as a resource, URL, or online content.
- The `vle` dataset complements `studentVle` by providing descriptive information about the resources students interact with.
- Together, `studentVle` and `vle` enable behavioral feature engineering based on different types of learning activities.

## understanding course.csv

In [35]:
courses.head()

,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268
3,BBB,2014J,262
4,BBB,2013B,240


In [36]:
courses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 3 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   code_module                 22 non-null     object
 1   code_presentation           22 non-null     object
 2   module_presentation_length  22 non-null     int64 
dtypes: int64(1), object(2)
memory usage: 660.0+ bytes


In [37]:
courses.describe(include="all")

,code_module,code_presentation,module_presentation_length
count,22,22,22.000000
unique,7,4,NaN
top,BBB,2014J,NaN
freq,4,7,NaN
mean,NaN,NaN,255.545455
std,NaN,NaN,13.654677
min,NaN,NaN,234.000000
25%,NaN,NaN,241.000000
50%,NaN,NaN,261.500000
75%,NaN,NaN,268.000000


### Key Findings from `courses`

- The `courses` dataset contains metadata about each course presentation.
- Each course is uniquely identified by the combination of `code_module` and `code_presentation`.
- The `module_presentation_length` column stores the total duration of each course in days.
- Different courses have different durations, which may be useful for time-based feature engineering in later stages.

# Conclusion

In this notebook, we explored all seven datasets of the OULAD project and understood their structure, relationships, and business meaning.

Key achievements:

- Loaded and explored all project datasets.
- Understood the role of each dataset in the online learning ecosystem.
- Identified the relationships between datasets using common keys.
- Detected potential data leakage candidates for further investigation.
- Identified behavioral, demographic, academic, and assessment-related features that may contribute to dropout prediction.

This analysis provides a strong foundation for the next stage of the project, where preprocessing, feature engineering, dataset merging, and model preparation will be performed.